# 2.8 多任务学习 (Multi-Task Learning)

同时学习多个相关任务，通过任务间的知识共享提升各任务的性能和泛化能力。

本节涵盖：
- 共享底座多任务
- 任务路由
- 梯度融合
- 联合训练

## 1. 共享底座多任务

**目的**：多任务共享参数以互相增益

**基本原理**：多个任务共享Transformer底座参数，不同任务通过不同的任务头或提示区分，共享的底座能学到更通用的表征，减少过拟合风险。

**优势**：
- 共享表征提供隐式的正则化效果
- 低资源任务可以从高资源任务的辅助中受益
- 单次推理可以同时服务多个任务

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SharedBackboneMTL(nn.Module):
    def __init__(self, input_dim=32, hidden=64, task_dims=[3, 5, 2]):
        super().__init__()
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU()
        )
        self.task_heads = nn.ModuleList([
            nn.Linear(hidden, dim) for dim in task_dims
        ])
        self.task_names = ['sentiment', 'topic', 'spam']

    def forward(self, x, task_id=None):
        shared = self.shared_encoder(x)
        if task_id is not None:
            return self.task_heads[task_id](shared)
        return [head(shared) for head in self.task_heads]

model = SharedBackboneMTL()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_samples = 200
X = torch.randn(n_samples, 32)
task_labels = [
    torch.randint(0, 3, (n_samples,)),
    torch.randint(0, 5, (n_samples,)),
    torch.randint(0, 2, (n_samples,)),
]

print('=== Shared Backbone Multi-Task Learning ===')
for epoch in range(30):
    all_logits = model(X)
    total_loss = 0
    for task_id, (logits, labels) in enumerate(zip(all_logits, task_labels)):
        loss = F.cross_entropy(logits, labels)
        total_loss += loss
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        accs = []
        for task_id, (logits, labels) in enumerate(zip(all_logits, task_labels)):
            acc = (logits.argmax(-1) == labels).float().mean().item()
            accs.append(acc)
        print(f'Epoch {epoch+1}: total_loss={total_loss.item():.4f}, accs={dict(zip(model.task_names, [f"{a:.3f}" for a in accs]))}')

shared_params = sum(p.numel() for p in model.shared_encoder.parameters())
head_params = sum(sum(p.numel() for p in head.parameters()) for head in model.task_heads)
print(f'\nShared params: {shared_params:,}, Head params: {head_params:,}')
print('Key: Shared encoder learns general features, task heads specialize.')

## 2. 任务路由（MoE风格）

**目的**：动态选择不同参数处理不同任务

**基本原理**：使用路由器根据输入动态激活不同的参数子集（如MoE中的专家），不同任务被路由到不同的专家组合，兼顾共享与特化。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class TaskRouterMoE(nn.Module):
    def __init__(self, input_dim=32, hidden=64, n_experts=4, n_classes=3, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = nn.Linear(input_dim, n_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, input_dim))
            for _ in range(n_experts)
        ])
        self.classifier = nn.Linear(input_dim, n_classes)

    def forward(self, x):
        router_logits = self.router(x)
        router_probs = F.softmax(router_logits, dim=-1)
        topk_probs, topk_indices = router_probs.topk(self.top_k, dim=-1)
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            expert_mask = (topk_indices == i).any(dim=-1)
            if expert_mask.any():
                expert_input = x[expert_mask]
                expert_output = expert(expert_input)
                for k in range(self.top_k):
                    selected = topk_indices[expert_mask, k] == i
                    if selected.any():
                        weight = topk_probs[expert_mask, k][selected].unsqueeze(-1)
                        idx = torch.where(expert_mask)[0][selected]
                        output[idx] += weight * expert_output[:selected.sum()]

        return self.classifier(output), router_probs

model = TaskRouterMoE()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_samples = 200
X = torch.randn(n_samples, 32)
y = torch.randint(0, 3, (n_samples,))

print('=== Task Router (MoE-style) Multi-Task Learning ===')
for epoch in range(30):
    logits, router_probs = model(X)
    cls_loss = F.cross_entropy(logits, y)
    load_balance_loss = n_experts * (router_probs.mean(0) ** 2).sum()
    loss = cls_loss + 0.01 * load_balance_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        acc = (logits.argmax(-1) == y).float().mean()
        expert_usage = router_probs.argmax(-1).bincount(minlength=n_experts).float() / n_samples
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, acc={acc:.3f}, expert_usage={expert_usage.tolist()}')

print('\nKey: Router dynamically selects experts for each input, enabling task specialization.')

## 3. 梯度融合（PCGrad）

**目的**：平衡多任务间的梯度冲突

**基本原理**：当不同任务的梯度方向冲突时，使用PCGrad将冲突梯度投影到另一梯度的法平面上，避免一个任务的更新损害另一个任务的性能。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def pcgrad_project(grad_i, grad_j):
    dot = torch.dot(grad_i, grad_j)
    if dot < 0:
        return grad_i - (dot / (torch.dot(grad_j, grad_j) + 1e-8)) * grad_j
    return grad_i

class TwoTaskModel(nn.Module):
    def __init__(self, input_dim=16, hidden=32):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU())
        self.head_a = nn.Linear(hidden, 3)
        self.head_b = nn.Linear(hidden, 2)
    def forward(self, x):
        shared = self.shared(x)
        return self.head_a(shared), self.head_b(shared)

model = TwoTaskModel()
X = torch.randn(100, 16)
y_a = torch.randint(0, 3, (100,))
y_b = torch.randint(0, 2, (100,))

print('=== Gradient Conflict Resolution (PCGrad) ===')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

conflict_count = 0
total_steps = 30
for step in range(total_steps):
    out_a, out_b = model(X)
    loss_a = F.cross_entropy(out_a, y_a)
    loss_b = F.cross_entropy(out_b, y_b)

    optimizer.zero_grad()
    loss_a.backward(retain_graph=True)
    grad_a = torch.cat([p.grad.clone().flatten() for p in model.parameters() if p.grad is not None])

    optimizer.zero_grad()
    loss_b.backward()
    grad_b = torch.cat([p.grad.clone().flatten() for p in model.parameters() if p.grad is not None])

    cos_sim = F.cosine_similarity(grad_a.unsqueeze(0), grad_b.unsqueeze(0)).item()
    if cos_sim < 0:
        conflict_count += 1

    grad_a_projected = pcgrad_project(grad_a, grad_b)
    grad_b_projected = pcgrad_project(grad_b, grad_a)

    merged_grad = (grad_a_projected + grad_b_projected) / 2
    idx = 0
    for p in model.parameters():
        size = p.numel()
        p.grad = merged_grad[idx:idx+size].reshape(p.shape)
        idx += size
    optimizer.step()

    if (step+1) % 10 == 0:
        with torch.no_grad():
            acc_a = (out_a.argmax(-1) == y_a).float().mean()
            acc_b = (out_b.argmax(-1) == y_b).float().mean()
        print(f'Step {step+1}: cos_sim={cos_sim:.3f}, acc_a={acc_a:.3f}, acc_b={acc_b:.3f}')

print(f'\nGradient conflicts detected: {conflict_count}/{total_steps} steps')
print('PCGrad projects conflicting gradients to reduce interference between tasks.')

## 4. 联合训练

**目的**：同时优化多个目标

**基本原理**：将多个任务的损失函数加权求和作为总损失，同时优化所有任务，权重可手动设定或自动学习，是大模型SFT阶段混合多种指令数据训练的常用方式。

**权重策略**：
- **等权重**：所有任务权重相同
- **按数据量加权**：大数据量任务获得更高权重
- **不确定性加权**：根据任务的不确定性自动调整权重（Kendall et al., 2018）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class UncertaintyWeightedMTL(nn.Module):
    def __init__(self, input_dim=16, hidden=32, n_tasks=3, task_outputs=[3, 5, 2]):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU())
        self.heads = nn.ModuleList([nn.Linear(hidden, d) for d in task_outputs])
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))
    def forward(self, x):
        shared = self.shared(x)
        return [head(shared) for head in self.heads]

model = UncertaintyWeightedMTL()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_samples = 200
X = torch.randn(n_samples, 16)
task_labels = [torch.randint(0, d, (n_samples,)) for d in [3, 5, 2]]
task_names = ['sentiment(3)', 'topic(5)', 'spam(2)']

print('=== Joint Training with Uncertainty Weighting ===')
for epoch in range(30):
    outputs = model(X)
    total_loss = 0
    task_losses = []
    for i, (logits, labels) in enumerate(zip(outputs, task_labels)):
        task_loss = F.cross_entropy(logits, labels)
        precision = torch.exp(-model.log_vars[i])
        weighted_loss = precision * task_loss + model.log_vars[i]
        total_loss += weighted_loss
        task_losses.append(task_loss.item())

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        weights = torch.exp(-model.log_vars).tolist()
        accs = [(logits.argmax(-1) == labels).float().mean().item() for logits, labels in zip(outputs, task_labels)]
        print(f'Epoch {epoch+1}: losses={[f"{l:.3f}" for l in task_losses]}, weights={[f"{w:.2f}" for w in weights]}')
        print(f'  accs={dict(zip(task_names, [f"{a:.3f}" for a in accs]))}')

print('\nKey: Uncertainty weighting automatically balances task losses.')
print('Tasks with higher uncertainty (log_var) get lower weight, preventing domination by hard tasks.')